# AML Transaction Monitoring Model
**Author:** Anushka Shinde | MS Finance, Boston University  




In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)

customer_types = ['Individual', 'Business', 'Shell Company', 'NGO']
countries = ['USA', 'UK', 'India', 'Cayman Islands', 'Panama',
             'Switzerland', 'Germany', 'UAE', 'Nigeria', 'Singapore']
high_risk_countries = ['Cayman Islands', 'Panama', 'Nigeria']
transaction_types = ['Wire Transfer', 'Cash Deposit', 'ATM Withdrawal',
                     'Online Transfer', 'Check', 'Crypto Exchange']

n = 500

df = pd.DataFrame({
    'Transaction_ID': [f'TXN{str(i).zfill(5)}' for i in range(1, n + 1)],
    'Customer_ID': [f'CUST{np.random.randint(1000, 2000)}' for _ in range(n)],
    'Customer_Type': np.random.choice(customer_types, n, p=[0.5, 0.3, 0.1, 0.1]),
    'Transaction_Amount': np.round(
        np.where(
            np.random.rand(n) > 0.95,
            np.random.uniform(9000, 9999, n),
            np.random.exponential(scale=3000, size=n).clip(100, 100000)
        ), 2),
    'Transaction_Type': np.random.choice(transaction_types, n),
    'Origin_Country': np.random.choice(countries, n,
        p=[0.3, 0.15, 0.15, 0.05, 0.05, 0.08, 0.1, 0.05, 0.04, 0.03]),
    'Destination_Country': np.random.choice(countries, n),
    'Num_Transactions_Last_30Days': np.random.randint(1, 50, n),
    'Avg_Transaction_Last_6Months': np.round(
        np.random.exponential(scale=2000, size=n).clip(100, 50000), 2),
    'Account_Age_Years': np.round(np.random.uniform(0.1, 20, n), 1),
    'Prior_SAR_Filed': np.random.choice([0, 1], n, p=[0.92, 0.08]),
})

print(f'Dataset ready — {len(df)} transactions loaded.')

Dataset ready — 500 transactions loaded.


In [ ]:
def apply_red_flags(row):
    """
    This function takes ONE transaction row as input.
    It checks it against 7 AML rules.
    It returns a string listing all the red flags found.
    If no flags, it returns 'None'.
    """

    # Start with an empty list — we'll add flags as we find them
    flags = []

    # --------------------------------------------------------
    # RULE 1: STRUCTURING
    # --------------------------------------------------------
    # Amounts between $9,000 and $9,999 are suspicious.
    # This is called structuring — deliberately keeping amounts
    # just below $10,000 to avoid triggering a CTR filing.
    # It is a federal crime in the USA (31 U.S.C. § 5324).
    if 9000 <= row['Transaction_Amount'] <= 9999:
        flags.append('Structuring')

    # --------------------------------------------------------
    # RULE 2: HIGH-RISK COUNTRY
    # --------------------------------------------------------
    # If the money is coming FROM or going TO a high-risk country,
    # it needs extra scrutiny. These jurisdictions have weak AML
    # controls and are commonly used for layering.
    if row['Origin_Country'] in high_risk_countries or \
       row['Destination_Country'] in high_risk_countries:
        flags.append('High-Risk Country')

    # --------------------------------------------------------
    # RULE 3: UNUSUAL AMOUNT VS HISTORY
    # --------------------------------------------------------
    # If this transaction is MORE THAN 5x the customer's
    # average transaction, something unusual is happening.
    # Example: Customer usually sends $500, but today sends $8,000.
    # This is a classic placement or layering indicator.
    if row['Avg_Transaction_Last_6Months'] > 0:
        ratio = row['Transaction_Amount'] / row['Avg_Transaction_Last_6Months']
        if ratio > 5:
            flags.append('Unusual Amount vs History')

    # --------------------------------------------------------
    # RULE 4: HIGH-RISK ENTITY TYPE
    # --------------------------------------------------------
    # Shell companies and NGOs are commonly used to move
    # illicit funds. When they use wire transfers (which are
    # harder to trace), that combination is a strong red flag.
    if row['Customer_Type'] in ['Shell Company', 'NGO'] and \
       row['Transaction_Type'] == 'Wire Transfer':
        flags.append('High-Risk Entity Type')

    # --------------------------------------------------------
    # RULE 5: PRIOR SAR ON FILE
    # --------------------------------------------------------
    # If a SAR was already filed for this customer, any new
    # transaction from them carries elevated inherent risk.
    if row['Prior_SAR_Filed'] == 1:
        flags.append('Prior SAR on File')

    # --------------------------------------------------------
    # RULE 6: NEW ACCOUNT HIGH VALUE
    # --------------------------------------------------------
    # An account less than 1 year old making transactions
    # above $10,000 is suspicious. New accounts are often
    # opened specifically to move illicit funds quickly.
    if row['Account_Age_Years'] < 1 and row['Transaction_Amount'] > 10000:
        flags.append('New Account High Value')

    # --------------------------------------------------------
    # RULE 7: EXCESSIVE TRANSACTION FREQUENCY
    # --------------------------------------------------------
    # More than 30 transactions in 30 days is unusual for
    # most customers. This pattern is linked to SMURFING —
    # breaking one large amount into many small transactions
    # across multiple accounts to avoid detection.
    if row['Num_Transactions_Last_30Days'] > 30:
        flags.append('Excessive Transaction Frequency')

    # --------------------------------------------------------
    # RETURN THE RESULT
    # --------------------------------------------------------
    # If we found flags, join them with a semicolon separator.
    # If no flags were found, return 'None'.
    if len(flags) > 0:
        return '; '.join(flags)
    else:
        return 'None'

print('Red flag function defined — ready to apply!')

Red flag function defined — ready to apply!


In [ ]:
# Apply the red flag function to every row
df['Red_Flags'] = df.apply(apply_red_flags, axis=1)

# Count how many flags each transaction has
# This will be useful later for the ML model
df['Flag_Count'] = df['Red_Flags'].apply(
    lambda x: 0 if x == 'None' else len(x.split(';'))
)

print('Red flags applied to all transactions!')
print(f'\n   Transactions with at least 1 flag : {(df["Flag_Count"] > 0).sum()}')
print(f'   Transactions with no flags        : {(df["Flag_Count"] == 0).sum()}')

Red flags applied to all transactions!

   Transactions with at least 1 flag : 388
   Transactions with no flags        : 112


In [ ]:
# Preview flagged transactions
print('Sample of flagged transactions:')
df[df['Flag_Count'] > 0][['Transaction_ID', 'Customer_Type', 'Transaction_Amount',
                            'Origin_Country', 'Red_Flags', 'Flag_Count']].head(10)

Sample of flagged transactions:


,Transaction_ID,Customer_Type,Transaction_Amount,Origin_Country,Red_Flags,Flag_Count
0,TXN00001,NGO,1392.82,USA,Unusual Amount vs History,1
1,TXN00002,Individual,927.13,Panama,High-Risk Country; Prior SAR on File; Excessiv...,3
2,TXN00003,Individual,9325.60,USA,Structuring; High-Risk Country; Unusual Amount...,3
3,TXN00004,Business,2075.45,Switzerland,Excessive Transaction Frequency,1
5,TXN00006,Individual,484.87,Germany,High-Risk Country; Excessive Transaction Frequ...,2
6,TXN00007,Individual,408.15,India,High-Risk Country,1
7,TXN00008,Individual,5999.90,India,Unusual Amount vs History; Excessive Transacti...,2
8,TXN00009,Shell Company,837.07,USA,Prior SAR on File,1
10,TXN00011,Business,4993.04,USA,High-Risk Country,1
11,TXN00012,Business,2856.39,India,High-Risk Country; Excessive Transaction Frequ...,2


In [ ]:
# How many times does each red flag appear?
print('Red Flag Frequency:')

flag_types = [
    'Structuring',
    'High-Risk Country',
    'Unusual Amount vs History',
    'High-Risk Entity Type',
    'Prior SAR on File',
    'New Account High Value',
    'Excessive Transaction Frequency'
]

for flag in flag_types:
    count = df['Red_Flags'].str.contains(flag).sum()
    pct = round(count / len(df) * 100, 1)
    print(f'   {flag:<35} : {count:>4} transactions ({pct}%)')

Red Flag Frequency:
   Structuring                         :   33 transactions (6.6%)
   High-Risk Country                   :  194 transactions (38.8%)
   Unusual Amount vs History           :  118 transactions (23.6%)
   High-Risk Entity Type               :   16 transactions (3.2%)
   Prior SAR on File                   :   46 transactions (9.2%)
   New Account High Value              :    0 transactions (0.0%)
   Excessive Transaction Frequency     :  198 transactions (39.6%)


In [ ]:
# How many flags does a single transaction have at most?
print('Flag Count Distribution:')
print(df['Flag_Count'].value_counts().sort_index())

print(f'\n   Max flags on one transaction : {df["Flag_Count"].max()}')
print(f'   Avg flags (flagged only)     : {df[df["Flag_Count"]>0]["Flag_Count"].mean():.2f}')

Flag Count Distribution:
Flag_Count
0    112
1    217
2    131
3     35
4      4
5      1
Name: count, dtype: int64

   Max flags on one transaction : 5
   Avg flags (flagged only)     : 1.56


In [ ]:
# Show the most suspicious transactions (most flags)
print('Top 10 Most Flagged Transactions:')
df.sort_values('Flag_Count', ascending=False)[
    ['Transaction_ID', 'Customer_ID', 'Customer_Type',
     'Transaction_Amount', 'Origin_Country', 'Red_Flags', 'Flag_Count']
].head(10)

Top 10 Most Flagged Transactions:


,Transaction_ID,Customer_ID,Customer_Type,Transaction_Amount,Origin_Country,Red_Flags,Flag_Count
468,TXN00469,CUST1476,Business,9617.43,Panama,Structuring; High-Risk Country; Unusual Amount...,5
30,TXN00031,CUST1276,NGO,6605.26,UK,High-Risk Country; Unusual Amount vs History; ...,4
285,TXN00286,CUST1641,Shell Company,11814.61,Nigeria,High-Risk Country; Unusual Amount vs History; ...,4
250,TXN00251,CUST1410,NGO,9476.22,Singapore,Structuring; High-Risk Country; Unusual Amount...,4
160,TXN00161,CUST1047,Business,9224.48,India,Structuring; High-Risk Country; Unusual Amount...,4
438,TXN00439,CUST1669,Business,9477.76,USA,Structuring; High-Risk Country; Unusual Amount...,3
60,TXN00061,CUST1646,NGO,775.70,Nigeria,High-Risk Country; High-Risk Entity Type; Exce...,3
442,TXN00443,CUST1562,Individual,1596.97,Panama,High-Risk Country; Unusual Amount vs History; ...,3
437,TXN00438,CUST1749,Business,1481.71,UK,High-Risk Country; Prior SAR on File; Excessiv...,3
57,TXN00058,CUST1130,Individual,2168.70,Cayman Islands,High-Risk Country; Prior SAR on File; Excessiv...,3
